In [5]:
# Importando as bibliotecas necessárias
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 1. Definindo os caminhos
# Como o notebook está na pasta 'notebooks', usamos '../' para voltar uma pasta e acessar 'data'
path_customers = '../data/raw/olist_customers_dataset.csv'
path_orders = '../data/raw/olist_orders_dataset.csv'
path_items = '../data/raw/olist_order_items_dataset.csv'

# 2. Carregando os dados [cite: 6]
print("Carregando bases originais...")
df_customers = pd.read_csv(path_customers)
df_orders = pd.read_csv(path_orders)
df_items = pd.read_csv(path_items)

print(f"Clientes: {df_customers.shape[0]} linhas")
print(f"Pedidos: {df_orders.shape[0]} linhas")
print(f"Itens: {df_items.shape[0]} linhas\n")

# 3. Unificando as bases (Merge)
# Juntamos pedidos com clientes usando a chave 'customer_id'
df_merged = pd.merge(df_orders, df_customers, on='customer_id', how='inner')

# Juntamos o resultado com os itens usando a chave 'order_id'
df_master = pd.merge(df_merged, df_items, on='order_id', how='inner')

print(f"Dataset Unificado: {df_master.shape[0]} linhas e {df_master.shape[1]} colunas\n")

# 4. Verificando valores faltantes (Requisito da 1ª Etapa) [cite: 7]
print("Valores Nulos por Coluna:")
nulos = df_master.isnull().sum()
display(nulos[nulos > 0].sort_values(ascending=False))

Carregando bases originais...
Clientes: 99441 linhas
Pedidos: 99441 linhas
Itens: 112650 linhas

Dataset Unificado: 112650 linhas e 18 colunas

Valores Nulos por Coluna:


order_delivered_customer_date    2454
order_delivered_carrier_date     1194
order_approved_at                  15
dtype: int64

In [6]:
# 5. Tratamento de Valores Faltantes (Limpeza)
print("Limpando dados nulos (pedidos não entregues/cancelados)...")
df_clean = df_master.dropna(subset=[
    'order_delivered_customer_date', 
    'order_delivered_carrier_date', 
    'order_approved_at'
])
print(f"Dataset após limpeza: {df_clean.shape[0]} linhas\n")

# 6. Selecionando variáveis numéricas iniciais para o Cluster
# Vamos focar no valor do produto (price) e no valor do frete (freight_value)
features = ['price', 'freight_value']
df_features = df_clean[features]

# 7. Escalonamento e Normalização 
from sklearn.preprocessing import MinMaxScaler, StandardScaler

print("Aplicando MinMaxScaler e StandardScaler...\n")

# MinMaxScaler (Comprime os dados para uma escala entre 0 e 1)
scaler_minmax = MinMaxScaler()
df_minmax = pd.DataFrame(scaler_minmax.fit_transform(df_features), columns=features)

# StandardScaler (Centraliza a média em 0 e o desvio padrão em 1)
scaler_standard = StandardScaler()
df_standard = pd.DataFrame(scaler_standard.fit_transform(df_features), columns=features)

# Exibindo a comparação exigida para a apresentação
print("=== Comparação das Escalas (Variável: Price) ===")
print(f"Original - Mínimo: {df_features['price'].min():.2f} | Máximo: {df_features['price'].max():.2f}")
print(f"MinMax   - Mínimo: {df_minmax['price'].min():.2f} | Máximo: {df_minmax['price'].max():.2f}")
print(f"Standard - Média:  {df_standard['price'].mean():.0f} | Desvio Padrão: {df_standard['price'].std():.0f}")

Limpando dados nulos (pedidos não entregues/cancelados)...
Dataset após limpeza: 110180 linhas

Aplicando MinMaxScaler e StandardScaler...

=== Comparação das Escalas (Variável: Price) ===
Original - Mínimo: 0.85 | Máximo: 6735.00
MinMax   - Mínimo: 0.00 | Máximo: 1.00
Standard - Média:  0 | Desvio Padrão: 1
